In [ ]:
!pip install tushare
import tushare as ts
pro=ts.pro_api('190a851c3e0ea12c3e6685fcd2a5b264713989258fdfdcd4b31f9cc6')
df = pro.daily(ts_code='600519.SH',              # 第2步：下载数据（贵州茅台）
               start_date='20240101',            # 开始日期
               end_date='20260430')              # 结束日期

print(df.head())   
import tushare as ts
import pandas as pd

# 用你的 token（已确认有2000积分）
pro = ts.pro_api('190a851c3e0ea12c3e6685fcd2a5b264713989258fdfdcd4b31f9cc6')

# 获取贵州茅台日线数据
df = pro.daily(ts_code='600519.SH', start_date='20200101', end_date='20260430')

# 按日期升序排列
df = df.sort_values('trade_date').reset_index(drop=True)

# 计算5日和20日均线
df['SMA5'] = df['close'].rolling(5).mean()
df['SMA20'] = df['close'].rolling(20).mean()

# 生成信号：1=买入，-1=卖出，0=持有
df['signal'] = 0
df.loc[df['SMA5'] > df['SMA20'], 'signal'] = 1
df.loc[df['SMA5'] < df['SMA20'], 'signal'] = -1

# 找出买卖点（信号变化的位置）
df['position'] = df['signal'].diff()
buy_signals = df[df['position'] == 2]   # 从-1变成1，即上穿
sell_signals = df[df['position'] == -2]  # 从1变成-1，即下穿

print("数据获取成功，共", len(df), "条")
print("买入信号数:", len(buy_signals))
print("卖出信号数:", len(sell_signals))
print(df[['trade_date', 'close', 'SMA5', 'SMA20', 'signal']].head(10))

import matplotlib.pyplot as plt
# 解决中文乱码配置（加在这里）
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'KaiTi']
plt.rcParams['axes.unicode_minus'] = False
# 计算策略收益率
df['returns'] = df['close'].pct_change()  # 每日涨跌幅
df['strategy_returns'] = df['signal'].shift(1) * df['returns']  # 次日按信号交易

# 累计收益
df['cumulative_strategy'] = (1 + df['strategy_returns']).cumprod()
df['cumulative_buyhold'] = (1 + df['returns']).cumprod()

# 打印收益率
strategy_return = (df['cumulative_strategy'].iloc[-1] - 1) * 100
buyhold_return = (df['cumulative_buyhold'].iloc[-1] - 1) * 100
print(f"策略总收益率: {strategy_return:.2f}%")
print(f"买入持有收益率: {buyhold_return:.2f}%")

# 画图
plt.figure(figsize=(14, 10))

# 子图1：价格与买卖点
plt.subplot(2, 1, 1)
plt.plot(df['trade_date'], df['close'], label='收盘价', alpha=0.7, linewidth=1)
plt.plot(df['trade_date'], df['SMA5'], label='5日均线', linewidth=1)
plt.plot(df['trade_date'], df['SMA20'], label='20日均线', linewidth=1)
plt.scatter(buy_signals['trade_date'], buy_signals['close'], color='red', marker='^', s=80, label='买入信号')
plt.scatter(sell_signals['trade_date'], sell_signals['close'], color='green', marker='v', s=80, label='卖出信号')
plt.legend()
plt.title('贵州茅台 - 双均线策略买卖点')
plt.xticks(rotation=45)

# 子图2：收益曲线对比
plt.subplot(2, 1, 2)
plt.plot(df['trade_date'], df['cumulative_strategy'], label='策略收益', color='blue', linewidth=1.5)
plt.plot(df['trade_date'], df['cumulative_buyhold'], label='买入持有', color='gray', linestyle='--', linewidth=1.5)
plt.legend()
plt.title('累计收益对比')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()